# CPU full-session showcase

Notebook tuyến tính để thử API `he_looming_sdk` bằng OpenFHE CPU. Chạy lần lượt từ trên xuống; GPU không được dùng trong notebook này.

## Chuẩn bị kernel

Tạo environment từ terminal tại root của repository, rồi mở Jupyter bằng chính environment đó:

```bash
python3.12 -m venv .venv-he-cpu
source .venv-he-cpu/bin/activate
python -m pip install --upgrade pip
python -m pip install -e '.[cpu]' 'jupyterlab>=4,<5'
python -m jupyter lab
```

> Wheel OpenFHE hiện tại yêu cầu Python 3.12/Linux Ubuntu 24.04. Notebook không sửa được lỗi ABI `GLIBCXX` trên Google Colab.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

from he_sdk import HESession, __version__

print("he_sdk version:", __version__)

## 1. Tạo CPU session

In [ ]:
session = HESession.create(device="cpu")

left_values = [1.0, 2.0, 3.0, 4.0]
right_values = [10.0, 20.0, 30.0, 40.0]

print("selected backend:", session.capabilities.backend)
print("capabilities:", session.capabilities)
print("left input:", left_values)
print("right input:", right_values)

## 2. Mã hóa và giải mã

In [ ]:
left_ct = session.encrypt(left_values)
right_ct = session.encrypt(right_values)

print("encrypted left:", left_ct)
print("decrypted left:", session.decrypt(left_ct))

## 3. Add và subtract

In [ ]:
add_ct = session.add(left_ct, right_ct)
subtract_ct = session.subtract(left_ct, right_ct)

print("add expected:", [11.0, 22.0, 33.0, 44.0])
print("add decrypted:", session.decrypt(add_ct))
print("subtract expected:", [-9.0, -18.0, -27.0, -36.0])
print("subtract decrypted:", session.decrypt(subtract_ct))

## 4. Multiply và square

In [ ]:
multiply_ct = session.multiply(left_ct, right_ct)
square_ct = session.square(left_ct)

print("multiply expected:", [10.0, 40.0, 90.0, 160.0])
print("multiply decrypted:", session.decrypt(multiply_ct))
print("square expected:", [1.0, 4.0, 9.0, 16.0])
print("square decrypted:", session.decrypt(square_ct))

## 5. Sum, mean và variance

In [ ]:
sum_ct = session.sum(left_ct)
mean_ct = session.mean(left_ct)
variance_ct = session.variance(left_ct)

print("sum expected:", 10.0)
print("sum decrypted:", session.decrypt(sum_ct))
print("mean expected:", 2.5)
print("mean decrypted:", session.decrypt(mean_ct))
print("variance expected:", 1.25)
print("variance decrypted:", session.decrypt(variance_ct))

## 6. Save và load ciphertext

Workspace nằm trong `generated/`, là thư mục đã được git ignore. Secret key không được ghi vào workspace.

In [ ]:
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
workspace = Path("generated") / f"cpu-sdk-showcase-{timestamp}"

session.save(left_ct, workspace, name="input")
session.save(sum_ct, workspace, name="sum")
loaded_input_ct = session.load(workspace, name="input")

print("workspace:", workspace.resolve())
print("loaded input decrypted:", session.decrypt(loaded_input_ct))

## 7. Phát hành kết quả cho recipient

In [ ]:
recipient = session.create_result_recipient()
recipient_directory = workspace / "recipient"
recipient.save_public_key(recipient_directory)

recipient_public_key = session.load_recipient_public_key(recipient_directory)
released_sum_ct = session.reencrypt_for_recipient(sum_ct, recipient_public_key)
session.save(released_sum_ct, workspace, name="released_sum")

recipient_sum_ct = recipient.load(workspace, name="released_sum")
print("recipient sum expected:", 10.0)
print("recipient sum decrypted:", recipient.decrypt(recipient_sum_ct))

## 8. Đóng owner session

In [ ]:
session.close()
print("owner session closed")

## 9. Mở compute-only session từ workspace

Compute session đọc public HE material và ciphertext đã lưu, thực hiện phép toán rồi lưu ciphertext kết quả. Session này không có owner secret key.

In [ ]:
compute = HESession.open_workspace(workspace)
compute_input_ct = compute.load(workspace, name="input")
compute_sum_ct = compute.sum(compute_input_ct)
compute.save(compute_sum_ct, workspace, name="compute_sum")

print("compute backend:", compute.capabilities.backend)
print("compute encrypted result:", compute_sum_ct)
compute.close()
print("compute session closed")